
# Exemplo Prático: Soma por **Divide-and-Conquer** no Modelo **Fork–Join** (Python)

Este notebook implementa, passo a passo, o exemplo dos slides:

- **Descrição do problema:** somar um vetor grande com paralelismo.
- **Estratégia adotada:** dividir recursivamente o problema em subproblemas menores (*divide-and-conquer*).
- **Fork:** criar subtarefas paralelas para subproblemas (as “folhas” da recursão).
- **Join:** sincronizar (aguardar) e **combinar** resultados parciais em uma *redução* (árvore de combinação).

> **Observação didática:**  
> Em Fork–Join, o paralelismo nasce da **divisão hierárquica** do problema e do **join** na volta (combinação).  
> Aqui representamos isso em Python com **ProcessPoolExecutor** e uma **árvore de redução**.

---

## Referências (para estudo)
- Cormen, T. et al. *Introduction to Algorithms* (3ª ed.). MIT Press, 2009.  
- McCool, M.; Robison, A.; Reinders, J. *Structured Parallel Programming*. Morgan Kaufmann, 2012.  
- Grama, A. et al. *Introduction to Parallel Computing* (2ª ed.). Addison-Wesley, 2003.



## 1) Imports e utilitários

Vamos usar:
- `concurrent.futures.ProcessPoolExecutor` para paralelismo por **processos**.
- `time.perf_counter()` para medir tempo.

Também criaremos funções para:
- gerar a árvore de divisão (intervalos)
- executar as folhas em paralelo (fork)
- combinar resultados em etapas (join)


In [15]:

from concurrent.futures import ProcessPoolExecutor, as_completed
import time
from typing import List, Tuple, Dict



## 2) Divisão do problema: construindo a árvore de subproblemas

No *divide-and-conquer*, um intervalo `[ini, fim)` é dividido em dois subintervalos até ficar pequeno o suficiente (limiar).

Vamos construir uma representação simples:
- **folhas**: intervalos pequenos que serão somados sequencialmente (base case)
- **nível (depth)**: profundidade na árvore (para visualizar o “hierárquico”)

Isso nos permite:
- mostrar como o Fork–Join “organiza” o paralelismo
- disparar (fork) as folhas em paralelo
- depois fazer join reduzindo os resultados


In [17]:

def dividir_intervalos(ini: int, fim: int, threshold: int, depth: int = 0,
                       folhas: List[Tuple[int,int,int]] = None) -> List[Tuple[int,int,int]]:
    """Retorna lista de folhas (ini, fim, depth) após dividir recursivamente até threshold."""
    if folhas is None:
        folhas = []
    if fim - ini <= threshold:
        print(ini, fim)
        folhas.append((ini, fim, depth))
        return folhas
    meio = (ini + fim) // 2
    dividir_intervalos(ini, meio, threshold, depth + 1, folhas)
    dividir_intervalos(meio, fim, threshold, depth + 1, folhas)
    return folhas

def mostrar_arvore_resumo(n: int, threshold: int) -> None:
    folhas = dividir_intervalos(0, n, threshold)
    depths = [d for _, _, d in folhas]
    print(f"n={n}, threshold={threshold}")
    print(f"Número de folhas (tarefas-base): {len(folhas)}")
    print(f"Profundidade máxima (aprox): {max(depths) if depths else 0}")
    hist: Dict[int,int] = {}
    for d in depths:
        hist[d] = hist.get(d, 0) + 1
    print("Folhas por profundidade:")
    for d in sorted(hist):
        print(f"  depth={d}: {hist[d]} folhas")



### Teste rápido: visualizando a divisão (vetor pequeno)


In [29]:

mostrar_arvore_resumo(n=10000, threshold=1000)


0 625
625 1250
1250 1875
1875 2500
2500 3125
3125 3750
3750 4375
4375 5000
5000 5625
5625 6250
6250 6875
6875 7500
7500 8125
8125 8750
8750 9375
9375 10000
n=10000, threshold=1000
Número de folhas (tarefas-base): 16
Profundidade máxima (aprox): 4
Folhas por profundidade:
  depth=4: 16 folhas



## 3) Base case: soma sequencial de uma folha

Cada folha `[ini, fim)` será somada sequencialmente (base case).  
O paralelismo aparece porque **várias folhas** são executadas em paralelo (fork), e depois combinadas (join).


In [19]:

def soma_folha(dados: List[int], ini: int, fim: int) -> int:
    """Soma sequencial da fatia dados[ini:fim]."""
    return sum(dados[ini:fim])



## 4) Fork: executando as folhas em paralelo

Aqui acontece o **fork**: criamos várias subtarefas independentes e submetemos ao executor.

> Nota: criamos **um único pool** e submetemos todas as folhas nele (boa prática para reduzir overhead).


In [20]:

def fork_folhas(dados: List[int], folhas: List[Tuple[int,int,int]], max_workers: int = None):
    """Executa folhas em paralelo e retorna lista de resultados."""
    resultados = []  # (ini, fim, depth, parcial)
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futuros = {ex.submit(soma_folha, dados, ini, fim): (ini, fim, depth) for ini, fim, depth in folhas}
        for fut in as_completed(futuros):
            ini, fim, depth = futuros[fut]
            parcial = fut.result()
            resultados.append((ini, fim, depth, parcial))
    resultados.sort(key=lambda x: x[0])
    return resultados



## 5) Join: combinando resultados em uma árvore de redução

Após obter as somas parciais, combinamos em níveis (redução em árvore), espelhando a hierarquia do Fork–Join.


In [21]:

def reduce_em_arvore(valores: List[int], verbose: bool = True) -> int:
    """Combina valores por soma em níveis (redução em árvore)."""
    nivel = 0
    atual = valores[:]
    if verbose:
        print(f"Nível {nivel}: {len(atual)} valores")

    while len(atual) > 1:
        prox = []
        it = iter(atual)
        for a in it:
            b = next(it, 0)  # se ímpar, soma com 0
            prox.append(a + b)
        atual = prox
        nivel += 1
        if verbose:
            print(f"Nível {nivel}: {len(atual)} valores")
    return atual[0] if atual else 0



## 6) Função principal: soma Fork–Join

Fluxo:
1. Divide → folhas  
2. Fork → executa folhas em paralelo  
3. Join → reduz em árvore


In [22]:

def soma_fork_join(dados: List[int], threshold: int, max_workers: int = None, verbose: bool = True) -> int:
    if threshold < 1:
        raise ValueError("threshold deve ser >= 1")
    n = len(dados)
    if n == 0:
        return 0

    folhas = dividir_intervalos(0, n, threshold)
    if verbose:
        print(f"Geradas {len(folhas)} folhas para n={n} com threshold={threshold}")

    resultados = fork_folhas(dados, folhas, max_workers=max_workers)

    if verbose:
        for ini, fim, depth, parcial in resultados[:min(8, len(resultados))]:
            print(f"folha [{ini}:{fim}] depth={depth} -> parcial={parcial}")
        if len(resultados) > 8:
            print("... (demais folhas omitidas)")

    parciais = [parcial for _, _, _, parcial in resultados]
    total = reduce_em_arvore(parciais, verbose=verbose)
    return total



## 7) Teste de corretude (vetor pequeno)


In [23]:

dados_pequenos = list(range(1, 21))
seq = sum(dados_pequenos)
fj = soma_fork_join(dados_pequenos, threshold=5, max_workers=4, verbose=True)

print("\nSequencial:", seq)
print("Fork–Join:", fj)
print("Corretude:", "OK" if seq == fj else "ERRO")


0 5
5 10
10 15
15 20
Geradas 4 folhas para n=20 com threshold=5
folha [0:5] depth=2 -> parcial=15
folha [5:10] depth=2 -> parcial=40
folha [10:15] depth=2 -> parcial=65
folha [15:20] depth=2 -> parcial=90
Nível 0: 4 valores
Nível 1: 2 valores
Nível 2: 1 valores

Sequencial: 210
Fork–Join: 210
Corretude: OK



## 8) Medição de tempo (didática)

> Em Python, paralelismo por processos tem overhead (IPC, serialização).  
> O objetivo aqui é **entender o modelo** e observar o trade-off entre `threshold` e desempenho.


In [24]:

def mede_tempo(func, *args, repeticoes: int = 3, **kwargs):
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        resultado = func(*args, **kwargs)
        t1 = time.perf_counter()
        tempos.append(t1 - t0)
    return resultado, min(tempos), tempos

def soma_fork_join_silenciosa(dados, threshold, max_workers):
    return soma_fork_join(dados, threshold=threshold, max_workers=max_workers, verbose=False)


In [25]:

N = 300_000  # ajuste conforme a máquina do laboratório
dados_grandes = list(range(1, N + 1))

seq_res, seq_best, seq_ts = mede_tempo(sum, dados_grandes, repeticoes=3)
print(f"Sequencial: resultado={seq_res}, melhor_tempo={seq_best:.4f}s, tempos={['%.4f' % t for t in seq_ts]}")

threshold = 50_000
max_workers = 4
fj_res, fj_best, fj_ts = mede_tempo(soma_fork_join_silenciosa, dados_grandes, threshold, max_workers, repeticoes=3)
print(f"Fork–Join: resultado={fj_res}, melhor_tempo={fj_best:.4f}s, tempos={['%.4f' % t for t in fj_ts]}")

print("Corretude:", "OK" if seq_res == fj_res else "ERRO")


Sequencial: resultado=45000150000, melhor_tempo=0.0016s, tempos=['0.0021', '0.0016', '0.0024']
0 37500
37500 75000
75000 112500
112500 150000
150000 187500
187500 225000
225000 262500
262500 300000
0 37500
37500 75000
75000 112500
112500 150000
150000 187500
187500 225000
225000 262500
262500 300000
0 37500
37500 75000
75000 112500
112500 150000
150000 187500
187500 225000
225000 262500
262500 300000
Fork–Join: resultado=45000150000, melhor_tempo=0.0631s, tempos=['0.0810', '0.0631', '0.0838']
Corretude: OK



## 9) Explorando thresholds (overhead vs paralelismo)

Vamos variar `threshold` e observar o efeito no tempo.


In [14]:

thresholds = [5_000, 10_000, 25_000, 50_000, 100_000]
max_workers = 4
tabela = []

for th in thresholds:
    res, t_best, _ = mede_tempo(soma_fork_join_silenciosa, dados_grandes, th, max_workers, repeticoes=3)
    tabela.append((th, t_best, res))

print("threshold | melhor_tempo(s) | resultado")
for th, t_best, res in tabela:
    print(f"{th:9d} | {t_best:14.4f} | {res}")


threshold | melhor_tempo(s) | resultado
     5000 |         0.2626 | 45000150000
    10000 |         0.1317 | 45000150000
    25000 |         0.0828 | 45000150000
    50000 |         0.0535 | 45000150000
   100000 |         0.0410 | 45000150000



## 10) Onde está o Fork–Join no código? (identificação conceitual)

**Fork (criação de tarefas):**
- acontece em `fork_folhas(...)` quando chamamos `executor.submit(...)` para cada folha.

**Join (sincronização):**
- `as_completed(...)` + `future.result()` garante que as folhas concluíram.

**Join (combinação hierárquica):**
- `reduce_em_arvore(...)` combina resultados em níveis, espelhando a árvore de divisão.

---

## Exercício para a turma (sugestão)
Implemente uma variação Fork–Join para calcular:
- o **máximo** do vetor (redução por `max`), ou
- a **contagem** de elementos que satisfazem uma condição (ex.: `x % 7 == 0`).

Requisitos:
- mesma divisão em folhas  
- fork nas folhas (paralelo)  
- join por redução em árvore  
